Merge ISPU Data and Formating

In [5]:
import pandas as pd
import os
import re

# CONFIG
FOLDER_PATH = "data/ISPU"
OUTPUT_PATH = "data/ISPU/ispu_dki_2010_2025.csv"

FILES_WITH_YEAR = {
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2024.csv": 2024,
    "data-indeks-standar-pencemar-udara-(ispu)-di-provinsi-dki-jakarta-komponen-data-2025.csv": 2025,
}

COLUMN_MAPPING = {
    "lokasi_spku": "stasiun",

    "pm_sepuluh": "pm10",
    "pm_10": "pm10",

    "pm_duakomalima": "pm2.5",
    "pm25": "pm2.5",

    "sulfur_dioksida": "so2",
    "nitrogen_dioksida": "no2",
    "karbon_monoksida": "co",
    "ozon": "o3",

    "categori": "kategori",
    "parameter_pencemar_kritis": "critical"
}

REQUIRED_COLUMNS = ["pm10", "pm2.5", "so2", "no2", "co", "o3"]

# HELPERS
def clean_tanggal(x):
    if pd.isna(x):
        return pd.NaT

    try:
        x = float(x)
        if x > 30000:
            return pd.to_datetime(x, unit="D", origin="1899-12-30")
    except:
        pass

    return pd.to_datetime(x, dayfirst=True, errors="coerce")


def clean_stasiun(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).upper().strip()

    match = re.search(r"DKI\s*([1-5])", x)
    if match:
        return f"DKI{match.group(1)}"

    return pd.NA


def process_ispu(df: pd.DataFrame, year: int | None = None) -> pd.DataFrame:
    df = df.copy()

    df.columns = df.columns.str.lower().str.strip()
    df.rename(columns=COLUMN_MAPPING, inplace=True)

    if {"bulan", "tanggal"}.issubset(df.columns) and year is not None:
        df["year"] = year
        df.rename(columns={"bulan": "month", "tanggal": "day"}, inplace=True)

        df["tanggal"] = pd.to_datetime(
            df[["year", "month", "day"]],
            errors="coerce"
        )

        df.drop(columns=["year", "month", "day"], inplace=True)

    if "tanggal" in df.columns:
        df["tanggal"] = (
            df["tanggal"]
            .apply(clean_tanggal)
            .dt.date
        )

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    if "stasiun" in df.columns:
        df["stasiun"] = df["stasiun"].apply(clean_stasiun)

    return df


# MAIN PIPELINE
dfs = []

for file in os.listdir(FOLDER_PATH):
    if not file.endswith(".csv"):
        continue
    if file == os.path.basename(OUTPUT_PATH):
        continue  

    path = os.path.join(FOLDER_PATH, file)
    df_raw = pd.read_csv(path)

    year = FILES_WITH_YEAR.get(file)  
    df_clean = process_ispu(df_raw, year)
    dfs.append(df_clean)


# concat semua
df_all = pd.concat(dfs, ignore_index=True)

df_all["tanggal"] = pd.to_datetime(df_all["tanggal"], errors="coerce")

df_all = (
    df_all
    .sort_values(["tanggal", "stasiun"])
    .reset_index(drop=True)
)

df_all.to_csv(OUTPUT_PATH, index=False)

print("Selesai Merge ISPU")


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_16016\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_16016\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_16016\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")
C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_16016\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified

Selesai Merge ISPU


C:\Users\rovan wardana\AppData\Local\Temp\ipykernel_16016\2564853698.py:46: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  return pd.to_datetime(x, dayfirst=True, errors="coerce")


Merge Cuaca

In [4]:
from pathlib import Path

data_cuaca_path = Path("data/cuaca-harian")

dfs = []

for file_path in data_cuaca_path.glob("cuaca-harian-dki[1-5]-*.csv"):
    filename = file_path.stem
    stasiun = filename.split("-")[2].strip().upper()  # DKI1–DKI5

    df = pd.read_csv(file_path)
    df['time'] = pd.to_datetime(df['time'], errors='coerce')
    df['stasiun'] = stasiun

    dfs.append(df)

if not dfs:
    raise ValueError("Tidak ada file DKI1–DKI5 yang terbaca")

df_cuaca_all = pd.concat(dfs, ignore_index=True)

# sorting
stasiun_order = ['DKI1','DKI2','DKI3','DKI4','DKI5']
df_cuaca_all['stasiun'] = pd.Categorical(
    df_cuaca_all['stasiun'],
    categories=stasiun_order,
    ordered=True
)

df_cuaca_all = (
    df_cuaca_all
    .sort_values(['time', 'stasiun'])
    .reset_index(drop=True)
)

output_path = data_cuaca_path / "cuaca-harian-dki-gabungan.csv"
df_cuaca_all.to_csv(output_path, index=False)

print("Selesai. File tersimpan di:", output_path)

Selesai. File tersimpan di: data\cuaca-harian\cuaca-harian-dki-gabungan.csv
